In [1]:
import pickle
import time
import random
from openai import OpenAI
random.seed(0)

In [2]:
def process_compositions(tables_dict, captions_dict, api_key):
    '''
    Converts 2D list table into list of tuples format
    table = [["A", "B", "C"], 
             ["1", "2", "3"]]
    table_tuples = [("A", "B", "C"), ("1", "2", "3")]
    
    We format it this way because:
    - Maintains data structure more explicitly
    - Makes it easier to process structured data
    - Preserves relationship between elements in each row
    '''
    # client = OpenAI(api_key=api_key)
    client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1"  # DeepSeek's API endpoint
    )
    outputs = {}
    retry_count = 10
    wait_time = 3  # Seconds to wait between retries
    
    for key, table in tables_dict.items():
        pii, t_idx = key
        # caption = captions_dict[key]
        # Convert each row to tuple and create list of tuples
        table_tuples = [tuple(str(cell) for cell in row) for row in table]
        caption = captions_dict.get(key, "")
        abstract = abstract_dict.get(pii, "")
        
        example_table_1 = [('Property/Composition', '0.15K2O-0.85SiO2', '0.25K2O-0.75SiO2', '0.33K2O-0.67SiO2'),
         ('Shear modulus G(kb)', '266', '241', '231'),
         ('Bulk modulus K s (kb)', '310', '283', '273'),
         ("Young's modulus E(kb)", '620', '564', '540'),
                          ]

        example_output_1 = """Table-Type = 1.
        [[('0022309374900039_0_0_1_0', 'K2O', 0.15, 'mol'), ('0022309374900039_0_0_1_0', 'SiO2', 0.85, 'mol')], [('0022309374900039_0_0_2_0', 'K2O', 0.25, 'mol'), ('0022309374900039_0_0_2_0', 'SiO2', 0.75, 'mol')], [('0022309374900039_0_0_3_0', 'K2O', 0.33, 'mol'), ('0022309374900039_0_0_3_0', 'SiO2', 0.67, 'mol')]]"""
        
        example_table_2 = [('Sample', 'Composition (mol%)', 'Composition (mol%)', 'Composition (mol%)', 'Composition (mol%)', 'Composition (mol%)', 'Composition (mol%)', 'r(g/cm 3)+-0.002', 'M (g/mol)+-0.05', 'V (mol/cm)+-0.05', 'H v (kg/mm2)', 'T g  (degC)+-3', 'T x  (degC)+-3'), 
                           ('Sample', 'CaO', 'Al2O3', 'MgO', 'SiO2', 'Er2O3', 'Yb2O3', 'r(g/cm 3)+-0.002', 'M (g/mol)+-0.05', 'V (mol/cm)+-0.05', 'H v (kg/mm2)', 'T g  (degC)+-3', 'T x  (degC)+-3'), 
                           ('CASM', '58.11', '27.11', '6.89', '7.89', '-', '-', '2.928', '67.75', '23.14', '865+-25', '841', '1058'), 
                           ('CAYB1', '57.77', '27.15', '6.95', '7.96', '-', '0.17', '2.960', '68.33', '23.08', '837+-18', '813', '1029')
                          ]

        example_output_2 = """Table-Type = 2.
        [[('S0022309300001332_0_2_1_0_CASM', 'CaO', 58.11, 'mol'), ('S0022309300001332_0_2_1_0_CASM', 'Al2O3', 27.11, 'mol'), ('S0022309300001332_0_2_1_0_CASM', 'MgO', 6.89, 'mol'), ('S0022309300001332_0_2_1_0_CASM', 'SiO2', 7.89, 'mol')], [('S0022309300001332_0_3_1_0_CAYB1', 'CaO', 57.77, 'mol'), ('S0022309300001332_0_3_1_0_CAYB1', 'Al2O3', 27.15, 'mol'), ('S0022309300001332_0_3_1_0_CAYB1', 'MgO', 6.95, 'mol'), ('S0022309300001332_0_3_1_0_CAYB1', 'SiO2', 7.96, 'mol'), ('S0022309300001332_0_3_1_0_CAYB1', 'Yb2O3', 0.17, 'mol')]]"""
        
        
        example_table_3 = [('Composition x (mol%)', 'l c (UV) (nm) (+-0.1)', 'l c (IR) (mm) (+-0.1)'),
         ('20', '440.0', '14.9'),
         ('30', '438.6', '10.5'),
         ('40', '435.0', '14.0'),
                          ]

        example_output_3 = """Table-Type = 3.
        [[('S0022309300001113_0_1_0_0', 'Li2O', 0.2, 'mol'), ('S0022309300001113_0_1_0_0', 'Bi2O3', 0.4, 'mol'), ('S0022309300001113_0_1_0_0', 'PbO', 0.4, 'mol')], [('S0022309300001113_0_2_0_0', 'Li2O', 0.3, 'mol'), ('S0022309300001113_0_2_0_0', 'Bi2O3', 0.35, 'mol'), ('S0022309300001113_0_2_0_0', 'PbO', 0.35, 'mol')], [('S0022309300001113_0_3_0_0', 'Li2O', 0.4, 'mol'), ('S0022309300001113_0_3_0_0', 'Bi2O3', 0.3, 'mol'), ('S0022309300001113_0_3_0_0', 'PbO', 0.3, 'mol')]]"""
        
        example_table_4 = [('Soot sample', 'Bulk density (g/cm3)', 'Specific surface area (m2/g)', 'NH3 high temperature treatment', 'Nitrogen content (ppm)'),
         ('A', '0.20', '19', '50 vol.% NH3, 800degC, 25 h', '5400'),
         ('B', '0.27', '16', '50 vol.% NH3, 800degC, 25 h', '5200'),
         ('C', '0.35', '11', '50 vol.% NH3, 800degC, 25 h', '5000')
                          ]

        example_output_4 = """Table-Type = 0.
        []"""

        
        # Combine examples with their metadata
        examples = [
            {
                "pii": "0022309374900039",
                "t_idx": 0,
                "table": example_table_1,
                "output": example_output_1,
                "caption": "",
                "abstract": "The elastic moduli and their variation with pressure have been determined for three glass compositions in the K2O\ue5f8SiO2 glass system. The results may be summarized:\nAdditionally, the heat capacities of the three compositions were determined over the temperature range of 320 to 700 K, and were used together with the other data to evaluate the thermal Grüneisen ratios. This parameter was found to increase with increasing K2O concentration, with values of 0.48, 0.63 and 0.69 for the three compositions indicated above. For comparison, the acoustic Grüneisen ratios evaluated from the variation of the moduli with pressure were found to be negative and also to increase with increasing K2O concentration."
            },
            {
                "pii": "S0022309300001332",
                "t_idx": 0,
                "table": example_table_2,
                "output": example_output_2,
                "caption": "Composition and properties of low silica content calcium aluminosilicate glasses",
                "abstract": "In this work a series of Er2O3 and Yb2O3 doped and Er2O3–Yb2O3 co-doped low silica calcium aluminosilicate glasses have been melted at 1470°C under vacuum conditions. Measurements of optical absorption coefficient, mass density, refractive index, Vickers micro-hardness, glass transformation temperature (T                     g) and crystallization temperature (T                     x) have been carried out. The results showed that these glasses dissolved ∼1.5 mol% Er2O3 and ∼1.1 mol% Yb2O3 in their structure without devitrification and also that only small changes (∼10%) have been measured in their thermal, mechanical and optical properties."
            },
            {
                "pii": "S0022309300001113",
                "t_idx": 0,
                "table": example_table_3,
                "output": example_output_3,
                "caption": "Cut-off wavelengths in the UV and IR regions for xLi2O-(100-x)[Bi2O3-PbO] glasses",
                "abstract": "New glasses with high Li2O content have been obtained in the system Li2O–Bi2O3–PbO. The role of the different components in the glass formation has been explored from the density and thermal measurements. A wide transmitting window having sharp cut-off in both ultraviolet–visible (UV–VIS) and infrared (IR) regions for almost all compositions of these glasses has been observed, making them an appealing candidate for different spectral devices."
            },
            {
                "pii": "S0022309300000053",
                "t_idx": 0,
                "table": example_table_4,
                "output": example_output_4,
                "caption": "Characteristics of some silica soot samples and their nitriding behavior",
                "abstract": "Incorporation of nitrogen in silica glass was conducted by sintering porous bodies of silica soot with absorbed NH3 gas. The properties, such as high temperature viscosity, internal friction and Young’s modulus, were evaluated and compared with those of several kinds of pure silica glasses. The nitrogen incorporation increased viscosity and Young’s modulus. From all results on both nitrogen-incorporated and pure silica glass, a relationship between Young’s modulus and density was deduced, and also a linkage between viscosity and internal friction was derived. The high temperature glass-foaming phenomenon due to nitrogen incorporation was studied. It was found that hot-isostatic pressing (HIP) treatment is effective for inhibiting foaming."
            }
        ]

        examples_text = "\n\n\n".join([
    f"Example {i+1}, Table Caption: ({ex['caption']}):\n"
    f"PII: {ex['pii']}, Table Index: {ex['t_idx']}\n"
    f"Abstract: {ex['abstract']}\n"
    f"Table: {ex['table']}\n"
    f"Output: {ex['output']}"
    for i, ex in enumerate(examples)
    ])

    #Three line gap has been maintained between each examples
        
        
        prompt = f"""Extract composition data following exact format of example, return only the structured data.
If no composition data found, return empty list [].

Examples, table with metadata and desired output:
{examples_text}

ID construction instructions:
- For column-oriented tables:
  - Compositions are in columns; values are read across.
  - ID format: pii_t_idx_r_first_comp_col_0_Material_ID (if Material_ID present), else pii_t_idx_r_first_comp_col_0.
    - `r` is the row containing values.
    - `first_comp_col` is the first column index where composition starts (1 in Example 1).
- For row-oriented tables:
  - Compositions are in rows; values are read down.
  - ID format: pii_t_idx_first_comp_row_c_0_Material_ID (if Material_ID present), else pii_t_idx_first_comp_row_c_0.
    - `c` is the column containing values.
    - `first_comp_row` is the first row index where composition starts.
- Both row and column indices of the table start from 0.
  
- Table-Type Classification:
 - Table-Type = 1: Entire composition is within a single cell.
 - Table-Type = 2: Composition is split across multiple cells, each holding a constituent.
 - Table-Type = 3: Partial composition in the table; remaining details inferred from text.
 - Table-Type = 0: No composition data in the table.
- Use 'mol' for molar/atomic percentages, 'wt' for weight percent, and default to 'mol' if unspecified.
- Return only the "Table-Type= Predicted Table-Type number. ", followed by the structured data list, no additional text.
- Ensure the output is fully generated and does not get truncated.

- For this table: pii={pii}, t_idx={t_idx}, caption={caption}, abstract={abstract}.
Table to process from which composition is to be extracted in the desired format:
{table_tuples}

Note: Follow the same format as the given example. Each composition as (id, component, value, unit), with values as floats. Remove tuples with value = 0."""

        # model = genai.GenerativeModel("gemini-1.5-pro")

        for attempt in range(retry_count):
            try:
                response = client.chat.completions.create(
                    model="deepseek-chat",  # Official model name deepseek-reasoner, deepseek-chat
                    messages=[{"role": "system", "content": "As a materials science expert skilled in extracting information from tables, your job is to extract compositions of materials from tables in the instructed format. Return only the table-type followed by the structured data list, no additional text."},
                        {"role": "user", "content": prompt}],
                    temperature=0,
                    max_tokens=8192
                )
        
                outputs[key] = response.choices[0].message.content.strip()

                pickle.dump(outputs, open('composition_test_results_deepseek_temporary.pkl', 'wb'))
        
                break  # Exit loop if successful
        
            except Exception as e:
                if any(msg in str(e).lower() for msg in ["rate limit", "429", "connection", "timeout", "busy", "expecting value"]):
                    if attempt < retry_count - 1:
                        time.sleep(wait_time)
                        print(e)
                        print(f"Rate limit hit, retrying in {wait_time} seconds (attempt {attempt+1}/{retry_count})...") # Informative message
                    else:
                        outputs[key] = f"Error: Rate limit reached after {retry_count} attempts."
                        print(outputs[key])  # Print the final error message
                        break
                else:
                    outputs[key] = f"Error: {str(e)}"
                    print(outputs[key])  # Print the error for debugging
                    break  # Do not retry for other errors

            

    return outputs

In [3]:
# !ls

In [2]:
random.seed(0)
start_time = time.time()
# tables = pickle.load(open('val_tables_dict_min.pkl', 'rb'))
# captions = pickle.load(open('val_captions_dict_min.pkl', 'rb'))
# abstract_dict = pickle.load(open('abstract_val_mini.pkl', 'rb'))
tables = pickle.load(open('test_tables_dict.pkl', 'rb'))
captions = pickle.load(open('test_captions_dict.pkl', 'rb'))
abstract_dict = pickle.load(open('abstract_test.pkl', 'rb'))
print(len(tables), len(captions), len(abstract_dict.keys()))
api_key = ''
results = process_compositions(tables, captions, api_key)
end_time = time.time()
print(f'Time required for processing {len(tables)} tables is {end_time -  start_time} seconds')
# pickle.dump(results, open('composition_results.pkl', 'wb'))

In [5]:
pickle.dump(results, open('composition_test_results_deepseek_chat_v3.pkl', 'wb'))
print(len(results))

737


In [1]:
# print(results)